# Adding memory

This extended lesson continues the one-conversation example: save and load conversations, and switch their IDs. First we build memory manually and save it to JSON. Then we use LangGraph's built-in checkpointer.

[LangChain short term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from hashlib import sha256
from html import escape
from pathlib import Path
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import HTML, display
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage, messages_from_dict, messages_to_dict
from langgraph.checkpoint.memory import InMemorySaver
from model_config import get_json_model, get_model, get_model_name

In [ ]:
model = get_model()
model_name = get_model_name()

COLORS = ["#2563eb", "#7c3aed", "#db2777", "#c2410c", "#15803d"]


def display_chat_message(chat_id: str, label: str, content: object) -> None:
    color_index = int(sha256(chat_id.encode()).hexdigest()[:8], 16) % len(COLORS)
    color = COLORS[color_index]
    safe_content = escape(str(content)).replace("\n", "<br>")
    display(HTML(
        f'<p><strong style="color: {color}">[{escape(chat_id)}] {label}:</strong> '
        f'{safe_content}</p>'
    ))

### Manual implementation of memory system

This version stores each conversation in `03_memory/manual_memory.json`. The conversation ID selects one nested JSON object.

```json
{
  "my-first-chat": {
    "turns": [
      {"type": "human", "data": {"content": "My name is Marek."}},
      {"type": "ai", "data": {"content": "Nice to meet you, Marek."}}
    ]
  }
}
```

The JSON is converted to LangChain `HumanMessage` and `AIMessage` objects before every agent call.

In [ ]:
# VS Code may start Jupyter in the project root or in notebooks/.
NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "04_adding_memory.ipynb").exists() else ROOT / "notebooks" / "langchain"
MEMORY_DIR = NOTEBOOK_DIR / "03_memory"
MEMORY_DIR.mkdir(exist_ok=True)
MEMORY_FILE = MEMORY_DIR / "manual_memory.json"
conversation_id = "1"


def load_conversations() -> dict:
    if MEMORY_FILE.exists():
        return json.loads(MEMORY_FILE.read_text(encoding="utf-8"))
    return {}


def save_conversations(conversations: dict) -> None:
    MEMORY_FILE.write_text(
        json.dumps(conversations, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


conversations = load_conversations()


def select_manual_conversation(new_id: str) -> None:
    global conversation_id

    conversation_id = new_id.strip()
    if not conversation_id:
        raise ValueError("Conversation ID cannot be empty.")

    conversations.setdefault(conversation_id, {"turns": []})
    print(f"Using manual conversation: {conversation_id}")


select_manual_conversation(conversation_id)


def show_manual_conversation() -> None:
    saved_turns = conversations[conversation_id]["turns"]
    for message in messages_from_dict(saved_turns):
        label = "Answer" if isinstance(message, AIMessage) else "You"
        display_chat_message(conversation_id, label, message.content)

In [ ]:
manual_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful, concise assistant.",
)

In [ ]:
def ask_question_with_manual_memory(prompt: str) -> str:

    saved_turns = conversations[conversation_id]["turns"]
    
    turns = messages_from_dict(saved_turns)
    turns.append(HumanMessage(content=prompt))
    display_chat_message(conversation_id, "You", prompt)

    result = manual_agent.invoke({"messages": turns})
    answer = result["messages"][-1].content

    turns.append(AIMessage(content=answer))
    conversations[conversation_id]["turns"] = messages_to_dict(turns)
    save_conversations(conversations)

    display_chat_message(conversation_id, "Answer", answer)
    return answer

In [ ]:
while True:
    request = input("Manual: ask, /list, /show, /switch <id>, or exit: ")

    if request.strip().lower() in ["exit", "q", "quit", "stop"]:
        break

    if request.strip() == "/list":
        print(list(conversations) or ["No saved conversations yet."])
        continue

    if request.strip() == "/show":
        show_manual_conversation()
        continue

    if request.startswith("/switch "):
        select_manual_conversation(request.removeprefix("/switch "))
        continue

    ask_question_with_manual_memory(request)

### Memory with LangGraph config

This version lets `InMemorySaver` keep the messages. It also writes snapshots to `03_memory/auto_memory.json`, so the same `thread_id` can be restored after a kernel restart.

In [ ]:
memory = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You are a helpful, concise assistant.",
    checkpointer=memory,
)

AUTO_MEMORY_FILE = MEMORY_DIR / "auto_memory.json"


def load_auto_conversations() -> dict:
    if AUTO_MEMORY_FILE.exists():
        return json.loads(AUTO_MEMORY_FILE.read_text(encoding="utf-8"))
    return {}


def save_auto_conversations() -> None:
    AUTO_MEMORY_FILE.write_text(
        json.dumps(auto_conversations, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


thread_id = "1"
auto_conversations = load_auto_conversations()
known_thread_ids = set(auto_conversations) | {thread_id}
restored_thread_ids = set()


def save_auto_conversation() -> None:
    state = agent.get_state(config)
    auto_conversations[thread_id] = {
        "turns": messages_to_dict(state.values.get("messages", []))
    }
    restored_thread_ids.add(thread_id)
    save_auto_conversations()


def select_config_conversation(new_id: str) -> None:
    global thread_id, config

    thread_id = new_id.strip()
    if not thread_id:
        raise ValueError("Conversation ID cannot be empty.")

    known_thread_ids.add(thread_id)
    config = {"configurable": {"thread_id": thread_id}}

    if thread_id in auto_conversations and thread_id not in restored_thread_ids:
        saved_turns = auto_conversations[thread_id]["turns"]
        agent.update_state(config, {"messages": messages_from_dict(saved_turns)})
        restored_thread_ids.add(thread_id)
        print(f"Restored config conversation: {thread_id}")

    print(f"Using config conversation: {thread_id}")


select_config_conversation(thread_id)


def show_config_conversation() -> None:
    saved_turns = auto_conversations.get(thread_id, {"turns": []})["turns"]
    for message in messages_from_dict(saved_turns):
        label = "Answer" if message.type == "ai" else "You"
        display_chat_message(thread_id, label, message.content)

In [ ]:
def ask_question_with_config_memory(prompt: str) -> str:
    if len(prompt.strip()) <= 3:
        return "Invalid input prompt"

    display_chat_message(thread_id, "You", prompt)

    result = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config=config,
    )
    answer = result["messages"][-1].content
    save_auto_conversation()
    display_chat_message(thread_id, "Answer", answer)
    return answer

In [ ]:
while True:
    request = input("Config: ask, /list, /show, /switch <id>, or exit: ")

    if request.strip().lower() in ["exit", "q", "quit", "stop"]:
        break

    if request.strip() == "/list":
        print(sorted(known_thread_ids))
        continue

    if request.strip() == "/show":
        show_config_conversation()
        continue

    if request.startswith("/switch "):
        select_config_conversation(request.removeprefix("/switch "))
        continue

    ask_question_with_config_memory(request)